In [1]:
# Install only what we need + remove the conflicting torchao
!pip install -q peft accelerate bitsandbytes
!pip uninstall -y torchao -q
!pip install -q peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import json

print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu128
True


# Load base model + tokenizer

In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Run baseline ("before") examples

In [4]:
# Same test sentences we'll reuse after fine-tuning
test_sentences = [
    "yo can u check this out when u have time",
    "srry running a few late, traffic is bad",
    "can we move the deadline up a bit?",
    "somethin seems wrong here, mind taking a look?",
]

def generate_reply(model, tokenizer, casual_text):
    messages = [
        {"role": "system", "content": "Rewrite the user's message in a formal, professional tone."},
        {"role": "user", "content": casual_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== BEFORE FINE-TUNING ===")
for s in test_sentences:
    print(f"\nCasual: {s}")
    print(f"Model:  {generate_reply(model, tokenizer, s)}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== BEFORE FINE-TUNING ===

Casual: yo can u check this out when u have time
Model:  Could you please provide me with more details so that I may review and assist you further?

Casual: srry running a few late, traffic is bad
Model:  I apologize for the delay; traffic has been particularly congested recently.

Casual: can we move the deadline up a bit?
Model:  Could you please advise on whether it would be possible to extend the deadline by a few days?

Casual: somethin seems wrong here, mind taking a look?
Model:  Could you please provide more details on what is causing concern so that I may assist you better?


# Load the dataset & format it

In [5]:
with open("/kaggle/input/datasets/nabaanabeeh/formality-dataset/formality_dataset.json") as f:
    raw_data = json.load(f)

def format_example(example):
    messages = [
        {"role": "system", "content": "Rewrite the user's message in a formal, professional tone."},
        {"role": "user", "content": example["casual"]},
        {"role": "assistant", "content": example["formal"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_data = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")

Dataset size: 74


In [6]:
print(dataset[0]["text"])

<|im_start|>system
Rewrite the user's message in a formal, professional tone.<|im_end|>
<|im_start|>user
hey can u send me that file when u get a sec<|im_end|>
<|im_start|>assistant
Hello, could you please send me that file at your earliest convenience? I would appreciate it.<|im_end|>



# LoRA setup

In [7]:
lora_config = LoraConfig(
    r=16,                    # rank - controls size of the adapter (higher = more capacity, more params)
    lora_alpha=32,           # scaling factor, usually 2x the rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # which layers to adapt (attention layers)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


# Training

In [9]:
# training arguments (plain HF Trainer, no trl needed)
training_args = TrainingArguments(
    output_dir="./formality_lora",       # where checkpoints would save
    num_train_epochs=4,                  # passes over the small dataset
    per_device_train_batch_size=4,       # examples per step
    gradient_accumulation_steps=2,       # effective batch size = 4*2 = 8
    learning_rate=2e-4,                  # standard LR for LoRA
    logging_steps=5,                     # print loss every 5 steps
    save_strategy="no",                  # skip checkpoint saving, not needed
    fp16=True,                           # use half precision on GPU
    report_to="none",                    # disable wandb/logging integrations
)

In [10]:
# tokenize the dataset (plain Trainer needs pre-tokenized input_ids)
def tokenize_function(example):
    tokens = tokenizer(example["text"], truncation=True, max_length=256, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, remove_columns=["text"])

Map:   0%|          | 0/74 [00:00<?, ? examples/s]

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss
5,7.281254
10,1.256069
15,0.627309
20,0.464989
25,0.374901
30,0.311822
35,0.274170
40,0.231746


TrainOutput(global_step=40, training_loss=1.3527825742959976, metrics={'train_runtime': 32.6723, 'train_samples_per_second': 9.06, 'train_steps_per_second': 1.224, 'total_flos': 597735675789312.0, 'train_loss': 1.3527825742959976, 'epoch': 4.0})

# Run("after") examples

In [12]:
print("=== AFTER FINE-TUNING ===")
for s in test_sentences:
    print(f"\nCasual: {s}")
    print(f"Model:  {generate_reply(model, tokenizer, s)}")

=== AFTER FINE-TUNING ===

Casual: yo can u check this out when u have time
Model:  Good morning/afternoon/evening, I wanted to reach out and see if you could take a moment to review this document for me?

Casual: srry running a few late, traffic is bad
Model:  I apologize for the delay; traffic has been particularly challenging today.

Casual: can we move the deadline up a bit?
Model:  May I suggest that we consider extending the deadline by a few days?

Casual: somethin seems wrong here, mind taking a look?
Model:  Could you please take a moment to review this matter and provide any insights or assistance that may be necessary?


In [13]:
model.save_pretrained("./formality_lora_adapter")
tokenizer.save_pretrained("./formality_lora_adapter")

('./formality_lora_adapter/tokenizer_config.json',
 './formality_lora_adapter/chat_template.jinja',
 './formality_lora_adapter/tokenizer.json')

# Results 
Data used: 74 hand-written casual→formal workplace message pairs (Slack/email tone), covering requests, apologies, follow-ups, scheduling, and confirmations. Built manually to ensure a clean, consistent style contrast for demonstrating style transfer — chose a small curated set over a large scraped dataset since this is a narrow style-transfer task, not a knowledge-learning task.

Model: Qwen2.5-1.5B-Instruct, fine-tuned with LoRA (r=16, alpha=32, targeting attention projection layers). Only 0.28% of parameters were trained (~4.3M of 1.5B).

Result: Training loss dropped from 7.28 to 0.23 over 4 epochs. Before fine-tuning, the model responded inconsistently to casual text; after fine-tuning, it reliably rewrites messages into polished, professional English with expanded, natural phrasing.